In [ ]:
##Spatial Consistency Check

In [ ]:
#1.Representativeness of a Pair

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, asin, sqrt
import re

# =========================
# CONFIG (แก้ให้ตรงของคุณ)
# =========================
STATIONS_CSV = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-1-Elevation_2024.csv"          # ต้องมีคอลัมน์: station_id, lat, lon, elev_m
DAILY_DIR    = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"            # โฟลเดอร์ไฟล์รายวัน (เช่น 364 ไฟล์)
ID_COL       = "station_id"
DATE_COL     = "date"                             # ถ้าไม่มีในไฟล์ จะอ่านจากชื่อไฟล์
RAIN_COL     = "rain(mm)"                         # ชื่อคอลัมน์ปริมาณฝน
RAINY_DAY_THRESHOLD = 1.0                         # ใช้เช็ค "วันฝนร่วม"
D_MAX_KM     = 50.0                               # รัศมีอิทธิพล
R_MIN        = 70.0                               # เกณฑ์คัดผู้ช่วย (อาจใช้ใน step ถัดไป)
OUT_PATH     = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-2-step1_pairs.csv"

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * asin(sqrt(a))

def normalize_id(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True))

# =========================
# Load stations
# =========================
st = pd.read_csv(STATIONS_CSV)
for col in [ID_COL, "lat", "lon", "elev_m"]:
    if col not in st.columns:
        raise ValueError(f"{STATIONS_CSV} must contain column '{col}'")
st[ID_COL] = normalize_id(st[ID_COL])
st = st[[ID_COL, "lat", "lon", "elev_m"]].copy()

# =========================
# Load daily files -> long & wide
# =========================
rows = []
for fp in sorted(Path(DAILY_DIR).glob("*.csv")):
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        raise ValueError(f"{fp.name} ต้องมีคอลัมน์ {ID_COL}, {RAIN_COL}")
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = normalize_id(df["station_id"])

    # date: ใช้ในไฟล์ถ้ามี ไม่งั้นอ่านจากชื่อไฟล์
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan  # ค่าติดลบ => missing
    rows.append(df[["station_id", "date", "rain"]])

if not rows:
    raise RuntimeError("ไม่พบไฟล์รายวันในโฟลเดอร์")

daily = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values(["station_id", "date"])

# wide table เพื่อคำนวณสหสัมพันธ์
wide = daily.pivot(index="date", columns="station_id", values="rain").sort_index()

# =========================
# เตรียม “คู่ในรัศมี D”
# =========================
pairs = []
sid_list = st[ID_COL].tolist()
for i, si in st.iterrows():
    for j, sj in st.iterrows():
        if si[ID_COL] == sj[ID_COL]:
            continue
        d_km = haversine_km(si.lat, si.lon, sj.lat, sj.lon)
        if d_km <= D_MAX_KM + 1e-9:
            h_m = abs(si.elev_m - sj.elev_m)
            pairs.append((si[ID_COL], sj[ID_COL], d_km, h_m))

pairs_df = pd.DataFrame(pairs, columns=["cand", "aux", "d_km", "h_m"])
if pairs_df.empty:
    raise RuntimeError("ไม่มีคู่สถานีภายในรัศมี D_MAX_KM ที่กำหนด")

# =========================
# คำนวณ Ccorr ต่อคู่ (เพียร์สัน; เงื่อนไข >=25 วันฝนร่วม)
# =========================
def corr_and_joint_rainy(cand, aux):
    s1 = wide.get(cand)
    s2 = wide.get(aux)
    if s1 is None or s2 is None:
        return 0.0, 0
    df2 = pd.concat([s1, s2], axis=1, keys=["c", "a"]).dropna(how="any")
    joint_rainy = int(((df2["c"] >= RAINY_DAY_THRESHOLD) & (df2["a"] >= RAINY_DAY_THRESHOLD)).sum())
    if joint_rainy < 25:
        return 0.0, joint_rainy
    c = df2["c"].corr(df2["a"])
    if pd.isna(c) or c < 0:
        c = 0.0
    return float(max(0.0, min(1.0, c))), joint_rainy

corrs = []
for cand, aux in pairs_df[["cand", "aux"]].itertuples(index=False):
    c, n_joint = corr_and_joint_rainy(cand, aux)
    corrs.append((cand, aux, c, n_joint))

corr_df = pd.DataFrame(corrs, columns=["cand", "aux", "Ccorr", "joint_rainy_days_ge1mm"])
pairs_df = pairs_df.merge(corr_df, on=["cand", "aux"], how="left")

# =========================
# คำนวณ R ตามสมการ
# =========================
term_dist = ((D_MAX_KM - pairs_df["d_km"]) / D_MAX_KM).clip(lower=0, upper=1)
term_alt  = (0.5 ** (pairs_df["h_m"] / 500.0)).clip(lower=0, upper=1)  # 0.5^(h/500)
term_corr = pairs_df["Ccorr"].clip(lower=0, upper=1)

R = (100.0/3.0) * (term_dist + term_alt + term_corr)
pairs_df["term_dist"] = term_dist
pairs_df["term_alt"]  = term_alt
pairs_df["term_corr"] = term_corr
pairs_df["R"] = R.clip(lower=0, upper=100)

# (ตัวเลือก) บอกสถานะเทียบกับ R_MIN
pairs_df["use_as_aux"] = np.where(pairs_df["R"] >= R_MIN, "yes", "no")

# =========================
# Save
# =========================
pairs_out = pairs_df.sort_values(["cand", "R"], ascending=[True, False])
pairs_out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")
print(pairs_out.head(10))


In [ ]:
#2.Daily Difference and Monthly Threshold

In [ ]:
##2.1 Cm-Monthly Threshold

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG (แก้ให้ตรงของคุณ)
# =========================
PAIRS_PATH = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-2-step1_pairs.csv"    # ผลจาก Step R (มีคอลัมน์ cand, aux, R)
DAILY_DIR  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index/"     # โฟลเดอร์ไฟล์รายวัน (หลังตัด Q<50 ถ้าทำแล้ว)
ID_COL     = "station_id"
DATE_COL   = "date"                             # ถ้าไม่มีในไฟล์ จะอ่านจากชื่อไฟล์
RAIN_COL   = "rain(mm)"
R_MIN      = 70.0                               # ให้สอดคล้องกับ Step R
OUT_CM     = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-cm_table.csv"

# ตั้งค่าควอนไทล์และขั้นต่ำจำนวนจุดต่อเดือน
QTL        = 0.95
MIN_N_PER_MONTH = 30   # ถ้า < 30 จุด จะใช้ fallback

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def norm_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# =========================
# Load pairs (ใช้เฉพาะ R >= R_MIN)
# =========================
pairs = pd.read_csv(PAIRS_PATH)
for col in ["cand","aux","R"]:
    if col not in pairs.columns:
        raise ValueError(f"{PAIRS_PATH} must contain columns 'cand','aux','R'")
pairs["cand"] = norm_id(pairs["cand"])
pairs["aux"]  = norm_id(pairs["aux"])
pairs_use = pairs[pairs["R"] >= R_MIN][["cand","aux","R"]].copy()
if pairs_use.empty:
    raise RuntimeError(f"No pairs with R >= {R_MIN} in {PAIRS_PATH}")

# =========================
# Load daily data -> long
# =========================
rows = []
for fp in sorted(Path(DAILY_DIR).glob("*.csv")):
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        raise ValueError(f"{fp.name} must contain {ID_COL} and {RAIN_COL}")
    df = df.rename(columns={ID_COL:"station_id", RAIN_COL:"rain"})
    df["station_id"] = norm_id(df["station_id"])
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    rows.append(df[["station_id","date","rain"]])

if not rows:
    raise RuntimeError(f"No CSV files in {DAILY_DIR}")

daily = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily["date"]  = pd.to_datetime(daily["date"])
daily["month"] = daily["date"].dt.month
daily = daily.sort_values(["station_id","date"])

# เตรียมดึงซีรีส์ต่อสถานี
by_sta = {sid: g.set_index("date")[["rain","month"]].sort_index()
          for sid, g in daily.groupby("station_id")}

# =========================
# สร้าง z = DIF / ln(101 - R) สำหรับทุกวัน-ทุกคู่
# =========================
z_rows = []
for cand, sub in pairs_use.groupby("cand"):
    s_c = by_sta.get(cand)
    if s_c is None:
        continue
    for aux, R in sub[["aux","R"]].itertuples(index=False):
        s_a = by_sta.get(aux)
        if s_a is None:
            continue
        # เอาเฉพาะคอลัมน์ rain ของผู้ช่วย แล้วค่อย rename เป็น rain_aux
        s_a2 = s_a[["rain"]].rename(columns={"rain": "rain_aux"})
        both = s_c.join(s_a2, how="inner")   # ตอนนี้ไม่มีคอลัมน์ month ชนกันแล้ว
        both = both.dropna(subset=["rain", "rain_aux"])

        if both.empty:
            continue
        ppt_c = both["rain"].values
        ppt_a = both["rain_aux"].values
        av    = 0.5*(ppt_c + ppt_a)
        mx    = np.maximum(ppt_c, ppt_a)
        dif   = np.where((ppt_c==0) & (ppt_a==0), 0.0, np.abs(ppt_c - ppt_a)/av * mx)

        rterm = np.log(101.0 - float(R))  # คงที่ต่อคู่
        if not np.isfinite(rterm) or rterm <= 1e-9:
            # R ใกล้ 100 มากเกินไป -> rterm เล็กมาก ทำให้ z โตไร้เหตุผล ตัดทิ้ง
            continue

        z = dif / rterm
        mon = both["month"].values.astype(int)

        z_rows.append(pd.DataFrame({
            "month": mon,
            "z": z
        }))

z_df = (pd.concat(z_rows, ignore_index=True) if z_rows else
        pd.DataFrame(columns=["month","z"]))

if z_df.empty:
    raise RuntimeError("No overlapping days between candidate-aux pairs to compute z.")

# =========================
# คำนวณ Cm ต่อเดือน (Q95 ของ z, พร้อม fallback)
# =========================
cms = []
for m in range(1, 13):
    z_m = z_df.loc[z_df["month"]==m, "z"].replace([np.inf, -np.inf], np.nan).dropna()
    n   = int(z_m.size)
    if n == 0:
        cms.append((m, np.nan, n))
        continue
    # winsorize ปลายบนเบา ๆ หากจุดเยอะพอ
    if n >= 100:
        cap = np.nanquantile(z_m, 0.99)
        z_m = np.clip(z_m, None, cap)
    Cm = float(np.nanquantile(z_m, QTL)) if n >= MIN_N_PER_MONTH else float(np.nanmedian(z_m))
    cms.append((m, Cm, n))

cm_df = pd.DataFrame(cms, columns=["month","Cm","n_points"])

# Fallback: ถ้าเดือนใด Cm เป็น NaN ให้ถัวจากเพื่อนบ้าน หรือใช้ค่ากลางทั้งปี
if cm_df["Cm"].isna().any():
    global_med = float(np.nanmedian(z_df["z"]))
    for i, row in cm_df[cm_df["Cm"].isna()].iterrows():
        m = row["month"]
        # เพื่อนบ้านเดือนใกล้เคียง
        neigh = cm_df.loc[cm_df["month"].isin([((m-2)%12)+1, (m%12)+1]) & cm_df["Cm"].notna(), "Cm"]
        cm_df.loc[i, "Cm"] = float(neigh.mean()) if len(neigh)>0 else global_med

# เซฟเฉพาะ columns ที่ต้องใช้จริง
cm_df[["month","Cm"]].to_csv(OUT_CM, index=False)
print(f"Saved: {OUT_CM}")
print(cm_df)


In [ ]:
##2.2 Daily Difference 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG (แก้ให้ตรงของคุณ)
# =========================
PAIRS_PATH   = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-2-step1_pairs.csv"  # ผลจาก Step R
DAILY_DIR    = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"          # โฟลเดอร์ไฟล์รายวัน (หลังตัด Q<50 แล้ว)
CM_TABLE_CSV = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-cm_table.csv"     # ต้องมีคอลัมน์: month(1..12), Cm
ID_COL       = "station_id"
DATE_COL     = "date"                              # ถ้าไม่มีในไฟล์จะอ่านจากชื่อไฟล์
RAIN_COL     = "rain(mm)"
R_MIN        = 70.0                                # ให้สอดคล้องกับขั้น R
# ถ้าไฟล์ pairs มีคอลัมน์ joint_rainy_days_ge1mm และอยากกรองขั้นต่ำ ให้ใส่ค่าไว้ที่นี่ (ไม่มีก็ไม่ใช้)
JOINT_MIN    = 60

OUT_DIF      = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-1-rel_step2a_dif.csv"
OUT_T        = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-2-rel_step2b_T.csv"
OUT_FLAGS    = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-3-rel_step2c_flags.csv"

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    s = path.stem
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", s)
    if not m:
        return pd.NaT
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def norm_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

# =========================
# Load pairs (R >= R_MIN) + (optional) joint rainy filter
# =========================
pairs = pd.read_csv(PAIRS_PATH)
for col in ["cand", "aux", "R"]:
    if col not in pairs.columns:
        raise ValueError("pairs file must contain columns 'cand','aux','R'.")

pairs["cand"] = norm_id(pairs["cand"])
pairs["aux"]  = norm_id(pairs["aux"])
pairs["R"]    = pd.to_numeric(pairs["R"], errors="coerce")

pairs_use = pairs[pairs["R"] >= R_MIN].copy()

# ถ้ามีคอลัมน์ joint_rainy_days_ge1mm และตั้งค่า JOINT_MIN ให้กรองเพิ่ม
if "joint_rainy_days_ge1mm" in pairs_use.columns and JOINT_MIN is not None:
    pairs_use = pairs_use[pairs_use["joint_rainy_days_ge1mm"] >= JOINT_MIN].copy()

pairs_use = pairs_use[["cand", "aux", "R"]].dropna()
if pairs_use.empty:
    raise RuntimeError("No pairs remain after filtering (R and joint rainy days).")

# =========================
# Load Cm (เดือน -> Cm)
# =========================
cm = pd.read_csv(CM_TABLE_CSV)
if not {"month","Cm"}.issubset(set(cm.columns)):
    raise ValueError("Cm table must contain columns: 'month','Cm'.")

cm["month"] = pd.to_numeric(cm["month"], errors="coerce").astype(int)
cm["Cm"]    = pd.to_numeric(cm["Cm"], errors="coerce")
if cm["month"].nunique() < 12:
    print("[WARN] Cm table has fewer than 12 distinct months.")
CM_MAP = dict(zip(cm["month"], cm["Cm"]))

# =========================
# Load daily data (long table)
# =========================
rows = []
for fp in sorted(Path(DAILY_DIR).glob("*.csv")):
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        raise ValueError(f"{fp.name} must contain {ID_COL} and {RAIN_COL}")
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})
    df["station_id"] = norm_id(df["station_id"])
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan
    rows.append(df[["station_id", "date", "rain"]])

daily = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
daily["date"]  = pd.to_datetime(daily["date"])
daily["month"] = daily["date"].dt.month

# จัดการ duplicate (station_id, date): เก็บค่าล่าสุดที่ไม่เป็น NaN
daily = (daily.sort_values(["station_id", "date"])
               .drop_duplicates(subset=["station_id", "date"], keep="last"))

# =========================
# STEP 2a — DIF (daily, pairwise)
# =========================
# เตรียมซีรีส์รายสถานี (index=date, cols=['rain','month'])
g_by_sta = {
    sid: g.set_index("date")[["rain", "month"]].sort_index()
    for sid, g in daily.groupby("station_id")
}

dif_rows = []
for cand, sub in pairs_use.groupby("cand"):
    s_c = g_by_sta.get(cand)
    if s_c is None:
        continue
    for aux, R in sub[["aux", "R"]].itertuples(index=False):
        s_a = g_by_sta.get(aux)
        if s_a is None:
            continue

        # join เฉพาะคอลัมน์ rain ของผู้ช่วย เพื่อตัดปัญหา month ซ้ำ
        s_a2 = s_a[["rain"]].rename(columns={"rain": "rain_aux"})
        both = s_c.join(s_a2, how="inner")
        both = both.dropna(subset=["rain", "rain_aux"])
        if both.empty:
            continue

        ppt_c = both["rain"].values
        ppt_a = both["rain_aux"].values
        av    = 0.5 * (ppt_c + ppt_a)
        mx    = np.maximum(ppt_c, ppt_a)
        dif   = np.where((ppt_c == 0) & (ppt_a == 0),
                         0.0,
                         np.abs(ppt_c - ppt_a) / np.maximum(av, 1e-12) * mx)

        tmp = pd.DataFrame({
            "cand":  cand,
            "aux":   aux,
            "date":  both.index.values,
            "month": both["month"].values.astype(int),  # ใช้เดือนของ candidate
            "ppt_c": ppt_c,
            "ppt_a": ppt_a,
            "DIF":   dif
        })
        dif_rows.append(tmp)

dif_df = (pd.concat(dif_rows, ignore_index=True) if dif_rows else
          pd.DataFrame(columns=["cand","aux","date","month","ppt_c","ppt_a","DIF"]))
dif_df = dif_df.sort_values(["cand", "date", "aux"])
dif_df.to_csv(OUT_DIF, index=False)
print(f"[2a] saved: {OUT_DIF} (rows={len(dif_df)})")

# =========================
# STEP 2b — T per pair-month (FULL GRID: ทุกคู่ × เดือน 1..12)
# =========================
# สร้างกริดคู่×เดือน ให้มี T ครบ 12 เดือนต่อทุกคู่ (จะไม่หายเดือน)
months = pd.DataFrame({"month": range(1, 13)})
pairs_use["_k"] = 1
months["_k"]    = 1
grid = pairs_use.merge(months, on="_k").drop(columns="_k")

# ผูก Cm และคำนวณ T
grid["Cm"] = grid["month"].map(CM_MAP).astype(float)
if grid["Cm"].isna().any():
    missing_months = sorted(grid.loc[grid["Cm"].isna(), "month"].unique())
    raise ValueError(f"Cm not found for months: {missing_months} in {CM_TABLE_CSV}")

grid["T"] = grid["Cm"] * np.log(101.0 - grid["R"].astype(float))
t_df = grid[["cand", "aux", "month", "R", "Cm", "T"]].sort_values(["cand","aux","month"])

t_df.to_csv(OUT_T, index=False)
print(f"[2b] saved: {OUT_T} (rows={len(t_df)})")

# =========================
# STEP 2c — Join DIF + T -> L flag (per day, per pair)
# =========================
if not dif_df.empty and not t_df.empty:
    flags = dif_df.merge(t_df, on=["cand","aux","month"], how="left")
    flags["L"] = np.where(flags["DIF"] < flags["T"], 1, 0).astype(int)
else:
    flags = pd.DataFrame(columns=list(dif_df.columns) + ["R","Cm","T","L"])

flags = flags.sort_values(["cand", "date", "aux"])
flags.to_csv(OUT_FLAGS, index=False)
print(f"[2c] saved: {OUT_FLAGS} (rows={len(flags)})")


In [ ]:
#3.Daily Weighted Agreement

In [ ]:
#3.Daily Weighted Agreement-Step1

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ========= CONFIG =========
FLAGS_PATH = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-3-3-rel_step2c_flags.csv"
DAILY_DIR  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"
ID_COL     = "station_id"
DATE_COL   = "date"
RAIN_COL   = "rain(mm)"
R_MIN      = 70.0
OUT_DAILY_LABELS = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-4-rel_step3_daily_labels.csv"

# 1) อ่าน flags (มีคอลัมน์ cand, date, aux, R, T, DIF, L)
flags = pd.read_csv(FLAGS_PATH, parse_dates=["date"])

# 2) รวมธงรายวันของ "สถานีเป้าหมาย" (cand) เป็น Wm + label
def agg_one(g):
    n_aux_used = g.shape[0]
    if n_aux_used < 3:
        return pd.Series({"Wm": np.nan, "label": "insufficient", "n_aux_used": n_aux_used})
    w = (g["R"] - R_MIN)**2
    den = w.sum()
    if den <= 0:
        return pd.Series({"Wm": np.nan, "label": "insufficient", "n_aux_used": n_aux_used})
    num = (g["L"] * w).sum()
    Wm = 100.0 * num / den
    if Wm >= 50.0:
        lab = "valid"
    elif Wm >= 20.0:
        lab = "doubtful"
    else:
        lab = "invalid"
    return pd.Series({"Wm": Wm, "label": lab, "n_aux_used": n_aux_used})

daily_labels = (flags.groupby(["cand","date"])
                      .apply(agg_one)
                      .reset_index()
                      .rename(columns={"cand":"station_id"}))

# 3) แนบค่าฝนของสถานีเป้าหมาย (ไว้ดู/ใช้ทำความสะอาดต่อ)
rows = []
for fp in sorted(Path(DAILY_DIR).glob("*.csv")):
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or RAIN_COL not in df.columns or DATE_COL not in df.columns:
        raise ValueError(f"{fp.name} must contain {ID_COL}, {DATE_COL}, {RAIN_COL}")
    sub = df[[ID_COL, DATE_COL, RAIN_COL]].copy()
    sub = sub.rename(columns={ID_COL:"station_id", DATE_COL:"date", RAIN_COL:"ppt"})
    sub["date"] = pd.to_datetime(sub["date"], errors="coerce")
    rows.append(sub)

daily_all = pd.concat(rows, ignore_index=True).dropna(subset=["date"])
out3 = (daily_labels.merge(daily_all, on=["station_id","date"], how="left")
                    .sort_values(["station_id","date"]))

out3.to_csv(OUT_DAILY_LABELS, index=False)
print(f"Saved: {OUT_DAILY_LABELS}")
print(out3.head())


In [ ]:
#3.Daily Weighted Agreement-Step2

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ========= CONFIG =========
DAILY_LABELS = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-4-rel_step3_daily_labels.csv"
DAILY_DIR_IN = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"
DAILY_DIR_OUT = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-7-QC_relative_cleaned"

ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# กติกาการทำความสะอาด (ปรับได้)
ACTION_INVALID      = "drop"   # "drop" (ลบแถว) หรือ "nan" (ตั้งค่าเป็น NaN)
ACTION_DOUBTFUL     = "keep"   # "keep" / "nan" / "drop"
ACTION_INSUFFICIENT = "keep"   # "keep" / "nan" / "drop"

Path(DAILY_DIR_OUT).mkdir(parents=True, exist_ok=True)

# 1) โหลดป้ายรายวันของสถานีเป้าหมาย
lab = pd.read_csv(DAILY_LABELS, parse_dates=["date"])
# ป้องกัน label เป็นค่าว่าง
lab["label"] = lab["label"].fillna("insufficient")

# 2) ฟังก์ชันตัดสินใจ
def apply_policy(df_day):
    # df_day: dataframe ของวันเดียว (ทุกสถานี)
    df = df_day.merge(lab, on=["station_id","date"], how="left")
    # ถ้าไม่มีผล label (เช่น สถานีนั้นไม่มีเพื่อนบ้าน) ถือว่า insufficient
    df["label"] = df["label"].fillna("insufficient")

    def do_action(mask, how):
        if how == "drop":
            return df.loc[~mask].copy()
        elif how == "nan":
            df.loc[mask, RAIN_COL] = np.nan
            return df
        else:  # keep
            return df

    # invalid
    df = do_action(df["label"]=="invalid", ACTION_INVALID)
    # doubtful
    df = do_action(df["label"]=="doubtful", ACTION_DOUBTFUL)
    # insufficient
    df = do_action(df["label"]=="insufficient", ACTION_INSUFFICIENT)

    # จัดคอลัมน์ผลลัพธ์สวย ๆ (แนบ Wm/label ไว้)
    keep_cols = [ID_COL, DATE_COL, RAIN_COL, "Wm", "label", "n_aux_used"]
    for c in ["Wm","n_aux_used"]:
        if c not in df.columns:
            df[c] = np.nan
    return df[keep_cols]

# 3) ประมวลผลทุกไฟล์รายวัน แล้วเซฟเป็นไฟล์ใหม่ชื่อเดิม
all_clean = []
for fp in sorted(Path(DAILY_DIR_IN).glob("*.csv")):
    df = pd.read_csv(fp)
    if ID_COL not in df.columns or DATE_COL not in df.columns or RAIN_COL not in df.columns:
        raise ValueError(f"{fp.name} must contain {ID_COL}, {DATE_COL}, {RAIN_COL}")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    # บางไฟล์อาจมีหลายวันรวมกัน ถ้าไฟล์คุณเป็น 1 วัน/ไฟล์ก็จะมีแค่กลุ่มเดียว
    out_parts = []
    for dt, g in df.groupby(df[DATE_COL].dt.normalize()):
        cleaned = apply_policy(g.copy())
        out_parts.append(cleaned)
        all_clean.append(cleaned)
    out_df = pd.concat(out_parts, ignore_index=True)
    out_df.to_csv(Path(DAILY_DIR_OUT) / fp.name, index=False)
    print(f"Cleaned -> {fp.name}")

# 4) สรุปรายปีต่อสถานี (%valid)
all_clean = pd.concat(all_clean, ignore_index=True)
all_clean["year"] = pd.to_datetime(all_clean[DATE_COL]).dt.year
summary = (all_clean
           .assign(is_valid = all_clean["label"].eq("valid").astype(int),
                   has_data = all_clean[RAIN_COL].notna().astype(int))
           .groupby(["station_id","year"])
           .agg(days_total=("has_data","sum"),
                days_valid=("is_valid","sum"))
           .reset_index())
summary["pct_valid"] = (summary["days_valid"] / summary["days_total"]).replace([np.inf, -np.inf], np.nan) * 100.0
summary = summary.sort_values(["year","station_id"])

SUM_OUT = Path(DAILY_DIR_OUT) / "relative_qc_summary_station_year.csv"
summary.to_csv(SUM_OUT, index=False)
print(f"Saved summary: {SUM_OUT}")
print(summary.head(10))

In [ ]:
#3.Daily Weighted Agreement-Step3

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ========= CONFIG (แก้ให้ตรงของคุณ) =========
DAILY_BASE = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"   # ใส่ 'โฟลเดอร์' ที่เก็บไฟล์รายวันหลัง Absolute QC (หรือใส่เป็น CSV เดี่ยวก็ได้)
LABELS     = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-4-rel_step3_daily_labels.csv"
OUT_BY_STATION_YEAR = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-5-summary_by_station_year.csv"
OUT_BY_STATION      = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-5-summary_by_station.csv"

ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"   # ถ้าชื่อคอลัมน์ฝนในไฟล์คุณเป็นอย่างอื่น ใส่ให้ตรง

# ========= helpers =========
def _parse_date_from_name(p: Path):
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", p.stem)
    return pd.NaT if not m else pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def _norm_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$","",regex=True)

# ========= 1) LOAD daily base (โฟลเดอร์หรือไฟล์เดียว) =========
base_rows = []
p = Path(DAILY_BASE)
if p.is_dir():
    fps = sorted(p.glob("*.csv"))
    if not fps:
        raise FileNotFoundError("ไม่พบไฟล์ CSV ในโฟลเดอร์ DAILY_BASE")
    for fp in fps:
        df = pd.read_csv(fp)
        if ID_COL not in df.columns or RAIN_COL not in df.columns:
            raise ValueError(f"{fp.name} ต้องมีคอลัมน์ {ID_COL} และ {RAIN_COL}")
        df = df.rename(columns={ID_COL:"station_id", RAIN_COL:"rain"})
        df["station_id"] = _norm_id(df["station_id"])
        if DATE_COL in df.columns:
            df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
        else:
            df["date"] = _parse_date_from_name(fp)
        df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
        df.loc[df["rain"] < 0, "rain"] = np.nan   # ค่าติดลบถือว่า missing
        base_rows.append(df[["station_id","date","rain"]])
    base = pd.concat(base_rows, ignore_index=True)
else:
    # เป็นไฟล์เดี่ยว
    base = pd.read_csv(p)
    base = base.rename(columns={ID_COL:"station_id", RAIN_COL:"rain"})
    base["station_id"] = _norm_id(base["station_id"])
    if DATE_COL in base.columns:
        base["date"] = pd.to_datetime(base[DATE_COL], errors="coerce")
    else:
        raise ValueError("ถ้า DAILY_BASE เป็นไฟล์เดี่ยว ต้องมีคอลัมน์ 'date'")
    base["rain"] = pd.to_numeric(base["rain"], errors="coerce")
    base.loc[base["rain"] < 0, "rain"] = np.nan

base = base.dropna(subset=["date"]).sort_values(["station_id","date"])
base["year"] = base["date"].dt.year
base["has_data"] = base["rain"].notna()

# ========= 2) LOAD labels และ LEFT JOIN เข้าฐาน =========
lab = pd.read_csv(LABELS, parse_dates=["date"])
lab["station_id"] = _norm_id(lab["station_id"])
for c in ("label","Wm","n_aux_used"):
    if c not in lab.columns:
        lab[c] = np.nan
lab["label"] = lab["label"].str.lower()

# left-join: คงสถานี–วันทั้งหมดจาก base
x = base.merge(lab[["station_id","date","label","Wm","n_aux_used"]],
               on=["station_id","date"], how="left")

# เติมค่า label/Wm/n_aux_used ที่ไม่มี => insufficient/NaN/0
x["label"] = x["label"].fillna("insufficient")
x["n_aux_used"] = pd.to_numeric(x["n_aux_used"], errors="coerce").fillna(0).astype(int)

# ========= DIAGNOSTICS =========
n_sta_base   = x["station_id"].nunique()
n_sta_labels = lab["station_id"].nunique()
print(f"Stations in BASE (after Absolute QC): {n_sta_base}")
print(f"Stations present in LABELS file     : {n_sta_labels}")
miss_in_labels = sorted(set(x["station_id"].unique()) - set(lab["station_id"].unique()))
print(f"Stations that had NO label rows at all (will be counted as 'insufficient'): {len(miss_in_labels)}")

# ========= 3) SUMMARY ต่อสถานี–ปี =========
g = x.copy()  # ใช้ฐานทั้งหมด; days_total จะนับเฉพาะวันมีฝนจริง
g["label"] = g["label"].fillna("insufficient").str.lower()

# นับแต่ละป้ายเฉพาะวันมีข้อมูลฝนจริง
g_use = g[g["has_data"]].copy()

counts = (g_use.groupby(["station_id","year","label"]).size()
          .unstack("label", fill_value=0)
          .reindex(columns=["valid","doubtful","invalid","insufficient"], fill_value=0)
          .reset_index())

# days_total จาก base (วันมีข้อมูลจริง)
days = (g.groupby(["station_id","year"])["has_data"].sum().reset_index(name="days_total"))

by_sy = counts.merge(days, on=["station_id","year"], how="outer").fillna(0)

by_sy = by_sy.rename(columns={
    "valid":"days_valid",
    "doubtful":"days_doubtful",
    "invalid":"days_invalid",
    "insufficient":"days_insufficient"
})
for c in ["days_valid","days_doubtful","days_invalid","days_insufficient","days_total"]:
    by_sy[c] = by_sy[c].astype(int)

def pct(df, num, den):
    return np.where(df[den]>0, 100*df[num]/df[den], np.nan)

by_sy["pct_valid"]        = pct(by_sy, "days_valid", "days_total")
by_sy["pct_doubtful"]     = pct(by_sy, "days_doubtful", "days_total")
by_sy["pct_invalid"]      = pct(by_sy, "days_invalid", "days_total")
by_sy["pct_insufficient"] = pct(by_sy, "days_insufficient", "days_total")
by_sy["pct_valid_doubt_insuf"] = by_sy[["pct_valid","pct_doubtful","pct_insufficient"]].sum(axis=1)

by_sy = by_sy.sort_values(["station_id","year"])
by_sy.to_csv(OUT_BY_STATION_YEAR, index=False)
print("Saved:", OUT_BY_STATION_YEAR)

# ========= 4) SUMMARY รวมทุกปีต่อสถานี =========
counts_s = (g_use.groupby(["station_id","label"]).size()
            .unstack("label", fill_value=0)
            .reindex(columns=["valid","doubtful","invalid","insufficient"], fill_value=0)
            .reset_index())
days_s = (g.groupby(["station_id"])["has_data"].sum().reset_index(name="days_total"))

by_s = counts_s.merge(days_s, on="station_id", how="outer").fillna(0).rename(columns={
    "valid":"days_valid","doubtful":"days_doubtful",
    "invalid":"days_invalid","insufficient":"days_insufficient"
})
for c in ["days_valid","days_doubtful","days_invalid","days_insufficient","days_total"]:
    by_s[c] = by_s[c].astype(int)

by_s["pct_valid"]        = pct(by_s, "days_valid", "days_total")
by_s["pct_doubtful"]     = pct(by_s, "days_doubtful", "days_total")
by_s["pct_invalid"]      = pct(by_s, "days_invalid", "days_total")
by_s["pct_insufficient"] = pct(by_s, "days_insufficient", "days_total")
by_s["pct_valid_doubt_insuf"] = by_s[["pct_valid","pct_doubtful","pct_insufficient"]].sum(axis=1)

by_s = by_s.sort_values(["station_id"])
by_s.to_csv(OUT_BY_STATION, index=False)
print("Saved:", OUT_BY_STATION)


In [ ]:
#3.Daily Weighted Agreement-Step4

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ================== CONFIG (แก้พาธให้ตรงของคุณ) ==================
DAILY_BASE = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index"                    # โฟลเดอร์ไฟล์รายวัน (หลังตัด Q<50 แล้ว)
LABELS     = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-4-rel_step3_daily_labels.csv" # ผล Wm+label รายวัน
OUT_LONG   = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-6-summary_overall_long.csv"   # ออกรูปแบบยาว: category, n_days, pct
OUT_WIDE   = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/3-6-summary_overall_wide.csv"   # ออกรูปแบบกว้างเหมือนตารางในเปเปอร์

ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"          # ชื่อคอลัมน์ฝนในไฟล์รายวันของคุณ

# ================== Helpers ==================
def parse_date_from_name(p: Path):
    m = re.search(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})", p.stem)
    return pd.NaT if not m else pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def norm_id(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$","",regex=True)

# ================== 1) LOAD daily base (รวมทุกไฟล์) ==================
p = Path(DAILY_BASE)
rows = []
if p.is_dir():
    for fp in sorted(p.glob("*.csv")):
        df = pd.read_csv(fp)
        if ID_COL not in df.columns or RAIN_COL not in df.columns:
            raise ValueError(f"{fp.name} ต้องมีคอลัมน์ {ID_COL} และ {RAIN_COL}")
        df = df.rename(columns={ID_COL:"station_id", RAIN_COL:"rain"})
        df["station_id"] = norm_id(df["station_id"])
        if DATE_COL in df.columns:
            df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
        else:
            df["date"] = parse_date_from_name(fp)
        df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
        df.loc[df["rain"] < 0, "rain"] = np.nan  # ค่าติดลบถือว่า missing
        rows.append(df[["station_id","date","rain"]])
    base = pd.concat(rows, ignore_index=True)
else:
    base = pd.read_csv(p)
    base = base.rename(columns={ID_COL:"station_id", RAIN_COL:"rain"})
    base["station_id"] = norm_id(base["station_id"])
    if DATE_COL not in base.columns:
        raise ValueError("ถ้า DAILY_BASE เป็นไฟล์เดี่ยว ต้องมีคอลัมน์ 'date'")
    base["date"] = pd.to_datetime(base["date"], errors="coerce")
    base["rain"] = pd.to_numeric(base["rain"], errors="coerce")
    base.loc[base["rain"] < 0, "rain"] = np.nan

base = base.dropna(subset=["date"]).sort_values(["station_id","date"])
base["has_data"] = base["rain"].notna()  # วันมีข้อมูลฝนจริง (0 มม. ถือว่ามีข้อมูล)

# ================== 2) LOAD labels และ left-join ==================
lab = pd.read_csv(LABELS, parse_dates=["date"])
lab["station_id"] = norm_id(lab["station_id"])
for c in ("label","Wm","n_aux_used"):
    if c not in lab.columns:
        lab[c] = np.nan
lab["label"] = lab["label"].str.lower()

x = base.merge(lab[["station_id","date","label","Wm","n_aux_used"]],
               on=["station_id","date"], how="left")

# label ที่ไม่มี (ไม่มีเพื่อนบ้านใช้งานได้วันนั้น) => insufficient
x["label"] = x["label"].fillna("insufficient")

# ================== 3) นับรวมทั้งเครือข่าย (เฉพาะวันมีข้อมูลจริง) ==================
use = x[x["has_data"]].copy()

order = ["valid","doubtful","invalid","insufficient"]
cnt = use["label"].value_counts().reindex(order, fill_value=0)
days_total = int(cnt.sum())

summary_long = (pd.DataFrame({"category": order, "n_days": cnt.values})
                .assign(pct=lambda d: np.where(days_total>0, 100*d["n_days"]/days_total, np.nan)))

# ================== 4) ออกรูปแบบ “กว้าง” เหมือน Table 3 ==================
# แถวเดียว: Total of days/station data, แล้วแต่ละหมวดมีทั้งจำนวนและ %
wide = {
    "total_days_station_data": days_total,
    "valid_n":        int(cnt["valid"]),
    "valid_pct":      (100*cnt["valid"]/days_total) if days_total else np.nan,
    "doubtful_n":     int(cnt["doubtful"]),
    "doubtful_pct":   (100*cnt["doubtful"]/days_total) if days_total else np.nan,
    "invalid_n":      int(cnt["invalid"]),
    "invalid_pct":    (100*cnt["invalid"]/days_total) if days_total else np.nan,
    "insufficient_n": int(cnt["insufficient"]),
    "insufficient_pct": (100*cnt["insufficient"]/days_total) if days_total else np.nan,
}
summary_wide = pd.DataFrame([wide])

# ================== 5) SAVE ==================
summary_long.to_csv(OUT_LONG, index=False)
summary_wide.to_csv(OUT_WIDE, index=False)

# (Optional) พิมพ์สรุปแบบสวย ๆ
def fmt_int(n):  return f"{n:,}"
def fmt_pct(p):  return ("{:.2f}%".format(p)).rjust(7)

print("\n=== OVERALL SUMMARY (days with data only) ===")
print("Total of days/station data:", fmt_int(days_total))
for cat in order:
    n = int(cnt[cat])
    p = 100*n/days_total if days_total else float("nan")
    print(f"{cat.capitalize():<12} {fmt_int(n):>12}  {fmt_pct(p)}")
